# Notebook 08 — Security and Unity Catalog Governance

**What you’ll learn:** Genie respects the **same** Unity Catalog access controls as any SQL tool. Column masks, row filters, and grants all apply — Genie is **not** a backdoor to your data.

**The scenario:** Your `production_events` table has a sensitive column called `unit_serial_vin` (Vehicle Identification Number). You want broad users to query production data through Genie, but you do **not** want them to see VIN values.

**What you’ll do:**
1. See real VIN values on the **base table** (admin view).
2. Build a **restricted view** that masks `unit_serial_vin` for non-admins using `is_member()`.
3. Compare direct SQL on the table **vs** the view.
4. Point a Genie space at **only** the restricted view.
5. **Try it yourself** — open the Genie UI and see the mask in action.
6. Verify programmatically via the Genie API.

**Key takeaway:** Unity Catalog governance is enforced regardless of how users access data — through SQL, dashboards, notebooks, or Genie.

**Before you start:** Run notebook **02** so `unit_serial_vin` exists in the data.

**Compute:** Serverless.

## Compute

Use **Serverless** (recommended).


## Your catalog and schema

Use the **same** widgets as **01**–**06**.


In [ ]:
%run ./00_workshop_config

## Step 1 — Sensitive data on the base table

Sample rows should show **populated** `unit_serial_vin` values (seeded in notebook **01**).


In [ ]:
display(
    spark.sql(
        f"""
SELECT event_id, event_type, event_date, part_number, unit_serial_vin
FROM {fqn}.production_events
WHERE unit_serial_vin IS NOT NULL
LIMIT 8
"""
    )
)


## Step 2 — Who has access to `production_events`?

Review grants before sharing any Genie space. Genie can only return data the backing tables/views allow.


In [ ]:
display(spark.sql(f"SHOW GRANTS ON TABLE {fqn}.production_events"))


## Step 3 — Audit log sample (Genie-related)

May require elevated workspace permissions; failures are expected in some sandboxes.


In [ ]:
try:
    display(
        spark.sql(
            """
        SELECT event_date, user_identity.email AS email, action_name
        FROM system.access.audit
        WHERE LOWER(CAST(action_name AS STRING)) LIKE '%genie%'
          AND event_date >= current_date() - INTERVAL 7 DAYS
        ORDER BY event_date DESC
        LIMIT 20
    """
        )
    )
except Exception as e:
    print("Audit query not available:", str(e)[:200])


## Step 4 — Restricted view (mask `unit_serial_vin`)

Non-members of **`admin_group`** see **NULL** for `unit_serial_vin`. Admins see the real value.

> Workshop tip: If you **are** in `admin_group`, you will see VINs in both the proof query and Genie — use a user **outside** that group to demo masking, or temporarily validate with `SELECT is_member('admin_group')`.


In [ ]:
spark.sql(
    f"""
CREATE OR REPLACE VIEW {fqn}.production_events_restricted AS
SELECT
  event_id,
  event_type,
  event_date,
  event_timestamp,
  production_line_id,
  operator_id,
  part_number,
  CASE WHEN is_member('admin_group') THEN unit_serial_vin ELSE CAST(NULL AS STRING) END AS unit_serial_vin,
  defect_code
FROM {fqn}.production_events
"""
)
print("Created view:", f"{fqn}.production_events_restricted")


## Step 5 — Direct SQL: base table vs restricted view

Same user, two objects: base table shows VINs; view should show **NULL** in `unit_serial_vin` when `is_member('admin_group')` is false.


In [ ]:
print("--- Base table (sensitive source) ---")
display(
    spark.sql(
        f"SELECT event_id, unit_serial_vin FROM {fqn}.production_events ORDER BY event_id LIMIT 5"
    )
)
print("--- Restricted view (what broad Genie users should see) ---")
display(
    spark.sql(
        f"SELECT event_id, unit_serial_vin FROM {fqn}.production_events_restricted ORDER BY event_id LIMIT 5"
    )
)
row = spark.sql(f"SELECT unit_serial_vin FROM {fqn}.production_events_restricted LIMIT 1").collect()
if row and row[0][0] is None:
    print("Masking check: first row unit_serial_vin is NULL (expected for non-admin_group).")
elif row:
    print("You appear to have admin_group (or equivalent) — VIN visible; use a non-admin principal to demo masking.")


## Step 6 — Genie space on the restricted view only

Creates a small **Data Room / Genie** space whose **only** table is **`production_events_restricted`**, so Genie cannot read the base table directly.

If the API returns an error, create the space manually in the UI and attach **only** this view.


In [ ]:
import re
import time

import requests
from databricks.sdk import WorkspaceClient

MASKED_SPACE_ID = None
w = WorkspaceClient()
host = w.config.host.rstrip("/")
api_headers = {**w.config.authenticate(), "Content-Type": "application/json"}

warehouses = list(w.warehouses.list())
warehouse_id = None
for wh in warehouses:
    state = str(wh.state).upper() if wh.state else ""
    if state in ("RUNNING", "STARTING") or getattr(wh, "enable_serverless_compute", False):
        warehouse_id = wh.id
        break
if not warehouse_id and warehouses:
    warehouse_id = warehouses[0].id

# Reuse existing masked-VIN space if it already exists; create only if needed
MASKED_SPACE_TITLE = f"{GENIE_SPACE_PREFIX} - Security Demo"
MASKED_SPACE_ID = None

existing = requests.get(f"{host}/api/2.0/genie/spaces", headers=api_headers)
if existing.status_code == 200:
    for s in existing.json().get("spaces", []):
        if s.get("title") == MASKED_SPACE_TITLE:
            MASKED_SPACE_ID = s.get("space_id") or s.get("id")
            print(f"Already exists: {MASKED_SPACE_TITLE!r} -> {MASKED_SPACE_ID}")
            break

if not MASKED_SPACE_ID and not warehouse_id:
    print("No SQL warehouse found — create the Genie space manually and attach", f"{fqn}.production_events_restricted")
elif not MASKED_SPACE_ID:
    payload = {
        "display_name": MASKED_SPACE_TITLE,
        "description": "Restricted view with column masking on VIN. Non-admins see null values.",
        "warehouse_id": warehouse_id,
        "table_identifiers": [f"{fqn}.production_events_restricted"],
        "run_as_type": "VIEWER",
    }
    resp = requests.post(f"{host}/api/2.0/data-rooms/", headers=api_headers, json=payload)
    if resp.status_code in (200, 201):
        body = resp.json()
        MASKED_SPACE_ID = body.get("space_id") or body.get("id")
        print("Created masked Genie space:", MASKED_SPACE_ID)
    else:
        print("Data Room API returned", resp.status_code, resp.text[:400])
        print("Manual: New Genie space → add only table", f"{fqn}.production_events_restricted")

if MASKED_SPACE_ID:
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    print("Open:", f"{host}/genie/rooms/{MASKED_SPACE_ID}{q}")


## Step 7 — Try it yourself: open the Genie space and ask for VINs

This is the most impactful part of the demo. Instead of running code, **you** (and every attendee) will open the masked Genie space in your browser and see the column mask live.

### Instructions

1. **Click the Genie space link** printed by the cell above.
2. In the Genie chat, type:
   > *Show me 8 production events from 2024 with event_id, event_type, and VIN, newest first.*
3. Look at the **unit_serial_vin** column in the result — it should show **null** or **[MASKED]** for every row.
4. **Facilitator (admin) comparison:** The facilitator runs the same question. If they are in `admin_group`, their result shows real VIN values — proving the mask is per-user, not per-query.

### What attendees should see

| Who | `unit_serial_vin` column | Why |
|-----|--------------------------|-----|
| Attendee (not in `admin_group`) | `null` everywhere | View masks the column via `CASE WHEN is_member(...)` |
| Facilitator (in `admin_group`) | Real VIN values | `is_member('admin_group')` returns true |

This is the same access control that applies to dashboards, notebooks, and any SQL tool — Genie does **not** bypass Unity Catalog.

In [ ]:
import html as _html

if MASKED_SPACE_ID:
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    url = f"{host}/genie/rooms/{MASKED_SPACE_ID}{q}"
    displayHTML(
        '<div style="background:#e8f5e9;border-left:4px solid #2e7d32;padding:16px;margin:8px 0;font-family:system-ui">'
        '<p style="font-size:16px;font-weight:bold;margin:0 0 8px 0">👉 Open this Genie space in your browser:</p>'
        f'<p style="margin:4px 0"><a href="{_html.escape(url, quote=True)}" target="_blank" rel="noopener" '
        f'style="font-size:15px">{_html.escape(url)}</a></p>'
        '<p style="margin:12px 0 4px 0"><b>Ask this question:</b></p>'
        '<p style="font-style:italic;margin:4px 0">"Show me 8 production events from 2024 with event_id, event_type, and VIN, newest first."</p>'
        '<p style="margin:12px 0 0 0">Check the <code>unit_serial_vin</code> column. '
        'If you are <b>not</b> in <code>admin_group</code>, every value should be <b>null</b>.</p>'
        '</div>'
    )
else:
    print("No masked space created. Run Step 6 first, or create one manually in the Genie UI.")


## Step 8 — Ask Genie for VINs and verify masking

If **`MASKED_SPACE_ID`** was created, we ask Genie to return VINs. For **non-admin** users, `unit_serial_vin` should be **NULL** in the result — proving Genie respects the view.


In [ ]:
# Programmatic verification: sends a question to Genie via API and checks the result
# This complements the interactive UI test in Step 7
def _ask_genie_show_vin_masking(space_id, question):
    if not space_id:
        print("No space id — skip (complete Step 6 or create space in UI).")
        return
    print("Asking:", question)
    start = requests.post(
        f"{host}/api/2.0/genie/spaces/{space_id}/start-conversation",
        headers=api_headers,
        json={"content": question},
    )
    if start.status_code != 200:
        print("start-conversation failed:", start.status_code, start.text[:300])
        return
    d = start.json()
    cid, mid = d.get("conversation_id"), d.get("message_id")
    for _ in range(30):
        time.sleep(3)
        poll = requests.get(
            f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}",
            headers=api_headers,
        )
        if poll.status_code != 200:
            continue
        msg = poll.json()
        st = msg.get("status", "")
        if st == "COMPLETED":
            for att in msg.get("attachments", []):
                aid = att.get("attachment_id") or att.get("id")
                if not att.get("query") or not aid:
                    continue
                qr = requests.get(
                    f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}/query-result/{aid}",
                    headers=api_headers,
                )
                if qr.status_code != 200:
                    continue
                j = qr.json()
                arr = j.get("statement_response", {}).get("result", {}).get("data_array", [])
                cols = j.get("statement_response", {}).get("manifest", {}).get("schema", {}).get("columns", [])
                names = [c.get("name", "") for c in cols]
                print("Columns:", names)
                for i, row in enumerate(arr[:8]):
                    print(f"Row {i+1}:", row)
                idx = [i for i, n in enumerate(names) if n and n.lower().replace("_", "") == "unitserialvin"]
                if not idx:
                    idx = [i for i, n in enumerate(names) if "vin" in n.lower() or "serial" in n.lower()]
                if idx and arr:
                    v = arr[0][idx[0]] if idx[0] < len(arr[0]) else None
                    if v is None or v == "":
                        print("MASKING VERIFIED via Genie: unit_serial_vin is null/empty in query result (non-admin view).")
                    else:
                        print("VIN visible in Genie result — if demoing masking, run as a user not in admin_group.")
                return
            print("Genie completed but no query attachment found.")
            return
        if st in ("FAILED", "CANCELLED"):
            print("Genie status:", st)
            return
    print("Timeout waiting for Genie.")


try:
    _ask_genie_show_vin_masking(
        MASKED_SPACE_ID,
        "Show event_id, event_type, and unit_serial_vin for 8 production events from 2024, newest first.",
    )
except NameError:
    print("Run the previous cell first to set MASKED_SPACE_ID / host / api_headers.")
except Exception as e:
    print("Genie check error:", str(e)[:200])


## Done — what you proved

| Check | Meaning |
|---|---|
| Base `production_events` | VINs exist (sensitive source). |
| `production_events_restricted` | Same rows, `unit_serial_vin` masked unless admin. |
| **Genie UI (interactive)** | Attendees see null VINs themselves — visual proof the mask works. |
| **Genie API (programmatic)** | Automated verification confirms masking via code. |
| **Admin vs non-admin** | Same question, different results — UC governs per-user access. |

**Takeaway:** Genie is **not** a backdoor. It reads data through Unity Catalog just like any other tool. Column masks, row filters, and grants all apply.

**Cleanup:** Delete the **Manufacturing workshop — Security demo** Genie space in the UI when finished if you do not want an extra space in the workspace.